# 203 — Report figures and original-MOU front comparison

This notebook makes polished report figures from the `201` and `202` outputs.

It also adds plots comparing the **original fully re-optimized 05_MOU fronts** for baseline, T −10%, and T +10%. Those original-front plots are different from the fixed-design re-evaluation plots:

- **Fixed-design re-evaluation:** same baseline Pareto designs, different T values.
- **Original MOU comparison:** separate optimized Pareto fronts created by running 05_MOU separately for each T value.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import uncertainty_project_helpers as up
up.apply_plot_style()

# -----------------------------
# User-editable settings
# -----------------------------
HISTORIC_STREAMFLOW_CFS = 8.6
PUMPING_COLUMN_FOR_HYDRO_PLOTS = "effective_total_pumping_cfs"

# If you want to tweak colors/labels, edit these dictionaries here.
SCENARIO_COLORS = up.SCENARIO_COLORS.copy()
SCENARIO_LABELS = up.SCENARIO_LABELS.copy()

# Original 05_MOU tradeoff CSVs. The helper searches this Uncertainty_Project folder,
# its parent notebooks folder, and LPR_pycap_opt/notebooks.
ORIGINAL_MOU_TRADEOFF_FILES = {
    "original_baseline": "fish_dollars_baseline_0.0_1.0_0.01_tradeoff.csv",
    "original_T_minus_10pct": "fish_dollars_Tmin10per_0.0_1.0_0.01_tradeoff.csv",
    "original_T_plus_10pct": "fish_dollars_Tplus10per_0.0_1.0_0.01_tradeoff.csv",
}

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = up.find_lpr_pycap_opt_dir(NOTEBOOK_DIR)
dirs = up.make_project_dirs(NOTEBOOK_DIR, "203", "uncertainty_report_figures")
OUTPUT_DIR = dirs["output_dir"]
PROJECT_OUTPUT_DIR = NOTEBOOK_DIR / "project_output"

OUT201 = PROJECT_OUTPUT_DIR / "201_reevaluate_baseline_pareto_under_T_scenarios"
OUT202 = PROJECT_OUTPUT_DIR / "202_probability_weighted_T_uncertainty_cost"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT201:", OUT201)
print("OUT202:", OUT202)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# -----------------------------
# Load 201 and 202 results
# -----------------------------
required_paths = [
    OUT201 / "201_T_scenario_reevaluation_wide.csv",
    OUT201 / "201_T_scenario_summary.csv",
    OUT202 / "202_probability_weighted_member_summary.csv",
    OUT202 / "202_probability_weighted_overall_summary.csv",
    OUT202 / "202_T_probability_scenario_summary.csv",
    OUT202 / "202_T_probability_weights.csv",
]
missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files. Run notebooks 201 and 202 first:
" + "
".join(str(p) for p in missing))

known_wide = pd.read_csv(OUT201 / "201_T_scenario_reevaluation_wide.csv")
known_summary = pd.read_csv(OUT201 / "201_T_scenario_summary.csv")
prob_members = pd.read_csv(OUT202 / "202_probability_weighted_member_summary.csv")
prob_overall = pd.read_csv(OUT202 / "202_probability_weighted_overall_summary.csv")
prob_scenarios = pd.read_csv(OUT202 / "202_T_probability_scenario_summary.csv")
prob_weights = pd.read_csv(OUT202 / "202_T_probability_weights.csv")

known_wide = known_wide.sort_values(PUMPING_COLUMN_FOR_HYDRO_PLOTS).reset_index(drop=True)
prob_members = prob_members.sort_values(PUMPING_COLUMN_FOR_HYDRO_PLOTS).reset_index(drop=True)

display(known_summary)
display(prob_overall)

In [ ]:
# -----------------------------
# Compact summary tables
# -----------------------------
def scalar_from_overall(metric_name):
    row = prob_overall.loc[prob_overall["metric"] == metric_name, "value"]
    return np.nan if row.empty else float(row.iloc[0])

baseline_row = known_summary.loc[known_summary["scenario"] == "baseline_T"].iloc[0]
tminus_row = known_summary.loc[known_summary["scenario"] == "T_minus_10pct"].iloc[0]
tplus_row = known_summary.loc[known_summary["scenario"] == "T_plus_10pct"].iloc[0]

compact_summary = pd.DataFrame([
    {
        "analysis": "Known error: T -10%",
        "T_value": tminus_row["T_value"],
        "mean_streamflow_cfs": tminus_row["mean_streamflow_cfs"],
        "mean_depletion_cfs": tminus_row["mean_depletion_cfs"],
        "mean_streamflow_change_from_baseline_cfs": tminus_row["mean_streamflow_change_from_baseline_cfs"],
        "mean_cost_metric_cfs": tminus_row["mean_streamflow_shortfall_below_baseline_cfs"],
        "max_cost_metric_cfs": tminus_row["max_streamflow_shortfall_below_baseline_cfs"],
    },
    {
        "analysis": "Known error: T +10%",
        "T_value": tplus_row["T_value"],
        "mean_streamflow_cfs": tplus_row["mean_streamflow_cfs"],
        "mean_depletion_cfs": tplus_row["mean_depletion_cfs"],
        "mean_streamflow_change_from_baseline_cfs": tplus_row["mean_streamflow_change_from_baseline_cfs"],
        "mean_cost_metric_cfs": tplus_row["mean_streamflow_shortfall_below_baseline_cfs"],
        "max_cost_metric_cfs": tplus_row["max_streamflow_shortfall_below_baseline_cfs"],
    },
    {
        "analysis": "Unknown T: probability-weighted absolute error",
        "T_value": np.nan,
        "mean_streamflow_cfs": np.nan,
        "mean_depletion_cfs": np.nan,
        "mean_streamflow_change_from_baseline_cfs": np.nan,
        "mean_cost_metric_cfs": scalar_from_overall("mean_probability_weighted_absolute_streamflow_error_cfs"),
        "max_cost_metric_cfs": scalar_from_overall("max_probability_weighted_absolute_streamflow_error_cfs"),
    },
    {
        "analysis": "Unknown T: probability-weighted shortfall",
        "T_value": np.nan,
        "mean_streamflow_cfs": np.nan,
        "mean_depletion_cfs": np.nan,
        "mean_streamflow_change_from_baseline_cfs": np.nan,
        "mean_cost_metric_cfs": scalar_from_overall("mean_probability_weighted_streamflow_shortfall_cfs"),
        "max_cost_metric_cfs": scalar_from_overall("max_probability_weighted_streamflow_shortfall_cfs"),
    },
])

display(compact_summary)
compact_summary.to_csv(OUTPUT_DIR / "203_compact_summary_table.csv", index=False)

In [ ]:
# -----------------------------
# Representative designs
# -----------------------------
def normalize_01(s):
    span = s.max() - s.min()
    if span == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / span

rep = prob_members.copy()
rep["pumping_norm"] = normalize_01(rep[PUMPING_COLUMN_FOR_HYDRO_PLOTS])
rep["streamflow_norm"] = normalize_01(rep["baseline_T_streamflow_cfs"])
rep["distance_to_ideal"] = np.sqrt((1 - rep["pumping_norm"])**2 + (1 - rep["streamflow_norm"])**2)

idxs = {
    "lowest_pumping": rep[PUMPING_COLUMN_FOR_HYDRO_PLOTS].idxmin(),
    "highest_pumping": rep[PUMPING_COLUMN_FOR_HYDRO_PLOTS].idxmax(),
    "highest_streamflow": rep["baseline_T_streamflow_cfs"].idxmax(),
    "lowest_streamflow": rep["baseline_T_streamflow_cfs"].idxmin(),
    "normalized_compromise_knee": rep["distance_to_ideal"].idxmin(),
}
representative_designs = []
for label, idx in idxs.items():
    row = rep.loc[idx].copy()
    row["representative_design"] = label
    representative_designs.append(row)
representative_designs = pd.DataFrame(representative_designs)

cols = [
    "representative_design", "member", PUMPING_COLUMN_FOR_HYDRO_PLOTS,
    "baseline_T_streamflow_cfs", "baseline_T_depletion_cfs",
    "expected_streamflow_cfs", "streamflow_std_cfs", "streamflow_p05_cfs", "streamflow_p95_cfs",
    "probability_weighted_absolute_streamflow_error_cfs",
    "probability_weighted_streamflow_shortfall_cfs",
    "max_streamflow_shortfall_below_baseline_cfs",
]
representative_designs = representative_designs[[c for c in cols if c in representative_designs.columns]]
display(representative_designs)
representative_designs.to_csv(OUTPUT_DIR / "203_representative_designs.csv", index=False)

In [ ]:
# -----------------------------
# Main report plots from fixed-design re-evaluation
# -----------------------------
up.plot_scenario_lines(
    known_wide,
    xcol=PUMPING_COLUMN_FOR_HYDRO_PLOTS,
    ycols_by_scenario={
        "T_minus_10pct": "streamflow_cfs__T_minus_10pct",
        "baseline_T": "streamflow_cfs__baseline_T",
        "T_plus_10pct": "streamflow_cfs__T_plus_10pct",
    },
    xlabel="Effective total pumping after fish-dollars cutoff (cfs)",
    ylabel="Streamflow = 8.6 cfs - depletion (cfs)",
    title="Fixed baseline designs re-evaluated under known T error",
    outfile=OUTPUT_DIR / "203_known_T_error_pumping_vs_streamflow.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)

p = up.sort_for_plot(prob_members, PUMPING_COLUMN_FOR_HYDRO_PLOTS)
fig, ax = plt.subplots()
ax.plot(p[PUMPING_COLUMN_FOR_HYDRO_PLOTS], p["baseline_T_streamflow_cfs"], linestyle="--", label="Baseline-T streamflow", color=SCENARIO_COLORS["baseline_T"])
ax.plot(p[PUMPING_COLUMN_FOR_HYDRO_PLOTS], p["expected_streamflow_cfs"], label="Expected streamflow", color=SCENARIO_COLORS["expected"])
ax.fill_between(p[PUMPING_COLUMN_FOR_HYDRO_PLOTS], p["streamflow_p05_cfs"], p["streamflow_p95_cfs"], alpha=0.22, color=SCENARIO_COLORS["uncertainty_band"], label="5th–95th percentile band")
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Streamflow = 8.6 cfs - depletion (cfs)")
ax.set_title("Unknown T: probability-weighted streamflow uncertainty")
ax.legend()
up.save_figure(fig, OUTPUT_DIR / "203_probability_weighted_expected_streamflow_band.png")

fig, ax = plt.subplots()
ax.plot(p[PUMPING_COLUMN_FOR_HYDRO_PLOTS], p["probability_weighted_streamflow_shortfall_cfs"], marker="o", markersize=3, color=SCENARIO_COLORS["T_plus_10pct"])
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted streamflow shortfall (cfs)")
ax.set_title("Cost of T uncertainty: one-sided streamflow shortfall")
up.save_figure(fig, OUTPUT_DIR / "203_probability_weighted_streamflow_shortfall.png")

fig, ax = plt.subplots()
ax.plot(p[PUMPING_COLUMN_FOR_HYDRO_PLOTS], p["probability_weighted_absolute_streamflow_error_cfs"], marker="o", markersize=3, color=SCENARIO_COLORS["expected"])
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted absolute streamflow error (cfs)")
ax.set_title("Cost of T uncertainty: absolute streamflow error")
up.save_figure(fig, OUTPUT_DIR / "203_probability_weighted_absolute_streamflow_error.png")

fig, ax = plt.subplots()
ax.bar(prob_weights["T_factor"], prob_weights["probability_weight"], width=0.025, color="0.45")
ax.set_xlabel("T factor relative to baseline")
ax.set_ylabel("Probability weight")
ax.set_title("Discrete probability weights for uncertain transmissivity")
up.save_figure(fig, OUTPUT_DIR / "203_T_probability_weights.png")

In [ ]:
# -----------------------------
# New: original fully optimized 05_MOU front comparison
# -----------------------------
original_tradeoffs = up.load_original_mou_tradeoff_fronts(
    project_dir=PROJECT_DIR,
    notebook_dir=NOTEBOOK_DIR,
    tradeoff_files=ORIGINAL_MOU_TRADEOFF_FILES,
)

print("Original tradeoff rows loaded:", len(original_tradeoffs))
if not original_tradeoffs.empty:
    display(original_tradeoffs.groupby("scenario").agg(
        n_points=("total_pumping_cfs", "count"),
        min_pumping_cfs=("total_pumping_cfs", "min"),
        max_pumping_cfs=("total_pumping_cfs", "max"),
        mean_streamflow_cfs=("streamflow_cfs", "mean"),
        mean_depletion_cfs=("depletion_cfs", "mean"),
    ).reset_index())
    original_tradeoffs.to_csv(OUTPUT_DIR / "203_original_MOU_tradeoff_fronts_loaded.csv", index=False)

up.plot_original_mou_tradeoffs(
    original_tradeoffs,
    ycol="streamflow_cfs",
    ylabel="Streamflow = 8.6 cfs - depletion (cfs)",
    title="Original 05_MOU optimized fronts: streamflow vs pumping",
    outfile=OUTPUT_DIR / "203_original_MOU_fronts_pumping_vs_streamflow.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)

up.plot_original_mou_tradeoffs(
    original_tradeoffs,
    ycol="depletion_cfs",
    ylabel="Streamflow depletion (cfs)",
    title="Original 05_MOU optimized fronts: depletion vs pumping",
    outfile=OUTPUT_DIR / "203_original_MOU_fronts_pumping_vs_depletion.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)

In [ ]:
# -----------------------------
# Write a short markdown results summary
# -----------------------------
mean_abs_error = scalar_from_overall("mean_probability_weighted_absolute_streamflow_error_cfs")
max_abs_error = scalar_from_overall("max_probability_weighted_absolute_streamflow_error_cfs")
mean_shortfall = scalar_from_overall("mean_probability_weighted_streamflow_shortfall_cfs")
max_shortfall = scalar_from_overall("max_probability_weighted_streamflow_shortfall_cfs")
mean_std = scalar_from_overall("mean_streamflow_std_cfs")
max_std = scalar_from_overall("max_streamflow_std_cfs")

summary_text = f"""# Project Results Summary — Transmissivity Uncertainty

## Fixed baseline Pareto designs under known T error

The baseline fish-dollars Pareto-front designs were held fixed and re-evaluated hydrologically.

| Scenario | T value | Mean streamflow (cfs) | Mean depletion (cfs) | Mean streamflow change from baseline (cfs) | Max absolute streamflow error (cfs) |
|---|---:|---:|---:|---:|---:|
| T -10% | {tminus_row['T_value']:.1f} | {tminus_row['mean_streamflow_cfs']:.6f} | {tminus_row['mean_depletion_cfs']:.6f} | {tminus_row['mean_streamflow_change_from_baseline_cfs']:.6f} | {tminus_row['max_absolute_streamflow_error_cfs']:.6f} |
| Baseline T | {baseline_row['T_value']:.1f} | {baseline_row['mean_streamflow_cfs']:.6f} | {baseline_row['mean_depletion_cfs']:.6f} | {baseline_row['mean_streamflow_change_from_baseline_cfs']:.6f} | {baseline_row['max_absolute_streamflow_error_cfs']:.6f} |
| T +10% | {tplus_row['T_value']:.1f} | {tplus_row['mean_streamflow_cfs']:.6f} | {tplus_row['mean_depletion_cfs']:.6f} | {tplus_row['mean_streamflow_change_from_baseline_cfs']:.6f} | {tplus_row['max_absolute_streamflow_error_cfs']:.6f} |

## Probability-weighted T uncertainty

- Mean probability-weighted absolute streamflow error: {mean_abs_error:.6f} cfs
- Max probability-weighted absolute streamflow error: {max_abs_error:.6f} cfs
- Mean probability-weighted streamflow shortfall: {mean_shortfall:.6f} cfs
- Max probability-weighted streamflow shortfall: {max_shortfall:.6f} cfs
- Mean streamflow standard deviation: {mean_std:.6f} cfs
- Max streamflow standard deviation: {max_std:.6f} cfs

## Interpretation

Higher transmissivity increases streamflow depletion and lowers streamflow for these fixed pumping designs. The one-sided shortfall metric is the most management-relevant cost metric because it focuses on cases where uncertain T causes actual streamflow to be lower than the baseline-T design predicted.
"""

summary_path = OUTPUT_DIR / "203_project_results_summary.md"
summary_path.write_text(summary_text)
print(summary_text)
print("Saved:", summary_path)

In [ ]:
print("All files saved in:")
print(OUTPUT_DIR)
print("
Files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(" -", f.name)